# MonthFu · Arabica monthly return forecasts

This notebook reads the completed **MonthFu saved artifacts**. Its default view does not retrain, refresh data or load a fitted model. The primary question is the return over **30 calendar days**, with four other fixed horizons for sensitivity.

**Cached origin at delivery: 9 September 2026.** This is the last local Yahoo price close, not today's market quote. Forecasts refer to that cached origin. Update the source caches and rerun training before treating a forecast as current.

A historical replay displays the **saved quarterly walk-forward score** for that origin. It never applies the final refitted weights to past dates.

**Execution status:** saved artifact checks and code syntax are verified. Notebook kernel execution was not completed in the managed sandbox because local Jupyter ports required permission. The cells remain ready to run in your normal notebook environment.


In [ ]:
from pathlib import Path
import json
import os

def find_repository():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "MonthFu" / "scripts" / "train.py").exists() and (folder / "data").exists():
            return folder
    raise FileNotFoundError("Open this notebook from the repository or its MonthFu folder.")

ROOT = find_repository()
MONTHFU = ROOT / "MonthFu"
ARTIFACTS = MONTHFU / "artifacts"
os.environ.setdefault("MPLCONFIGDIR", str(ARTIFACTS / ".matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

HORIZONS = {
    "30calendar": "30 calendar days · primary",
    "21sessions": "21 observed sessions",
    "28calendar": "28 calendar days · four weeks",
    "21calendar": "21 calendar days · three weeks",
    "30sessions": "30 observed sessions · about six weeks",
}
PRIMARY = "30calendar"
VIEW_EXPERIMENT = PRIMARY
REPLAY_DATE = None  # Example: "2024-06-03"; must be a saved evaluated origin.

required = [ARTIFACTS / "horizon_comparison.csv", ARTIFACTS / "latest_forecasts.csv"]
required += [ARTIFACTS / key / name for key in HORIZONS for name in
             ["metrics.json", "holdout_predictions.csv", "holdout_by_year.csv", "latest_forecast.csv"]]
missing = [str(path.relative_to(ROOT)) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Training artifacts are incomplete. Run the commands below. Missing: " + ", ".join(missing))

comparison = pd.read_csv(ARTIFACTS / "horizon_comparison.csv")
latest_forecasts = pd.read_csv(ARTIFACTS / "latest_forecasts.csv", parse_dates=["as_of_date", "target_date_estimate"])
summaries = {key: json.loads((ARTIFACTS / key / "metrics.json").read_text()) for key in HORIZONS}
holdouts = {key: pd.read_csv(ARTIFACTS / key / "holdout_predictions.csv",
                          parse_dates=["Date", "target_end_date"]) for key in HORIZONS}
yearly = {key: pd.read_csv(ARTIFACTS / key / "holdout_by_year.csv") for key in HORIZONS}
cached_origin = latest_forecasts.as_of_date.max()
assert latest_forecasts.as_of_date.nunique() == 1
assert set(comparison.experiment) == set(HORIZONS)
print(f"Loaded five saved experiments. Cached forecast origin: {cached_origin:%d %B %Y}.")


## Target definitions and availability

`target_return = Close[target_end_date] / Close[origin] - 1`.

| Target | Endpoint |
| --- | --- |
| **30 calendar days, primary** | First observed market close on or after origin + 30 calendar days |
| 21 observed sessions | Twenty-first subsequent observed market close |
| 28 calendar days | First observed close on or after origin + 28 calendar days |
| 21 calendar days | First observed close on or after origin + 21 calendar days |
| 30 observed sessions | Thirtieth subsequent observed market close |

COT positions describe a snapshot, typically Tuesday, and publish later. Publication occurs after the Coffee C close, so a report enters forecasts on the **first following observed session**. Weekly positioning changes are computed before daily forward filling. Release exceptions, outages, corrected archives and backcast history have separate guards; see [COT alignment](docs/COT_ALIGNMENT.md).

Weather uses calendar-day rollups and a five-calendar-day availability proxy. It is historical reanalysis without original publication vintages. News summaries were selected retrospectively and remain excluded from default model selection.

Daily monthly targets overlap. The quarterly walk-forward models use only labels matured before each refit, and the recipe is frozen using pre-holdout expanding validation. Historical holdout reuse and imperfect source vintages make these exploratory results.


In [ ]:
display(Markdown("### Compare fixed horizons"))
table = comparison.copy()
table["target"] = table.experiment.map(HORIZONS)
table = table[["target", "cv_skill_vs_zero", "holdout_rmse", "previous_rmse",
               "holdout_rmse_improvement_vs_previous", "direction_accuracy",
               "training_majority_accuracy", "common_holdout_rows"]]
display(table.style.format({
    "cv_skill_vs_zero": "{:+.2%}", "holdout_rmse": "{:.2%}", "previous_rmse": "{:.2%}",
    "holdout_rmse_improvement_vs_previous": "{:+.2%}", "direction_accuracy": "{:.2%}",
    "training_majority_accuracy": "{:.2%}", "common_holdout_rows": "{:,.0f}",
}).hide(axis="index"))
display(Markdown(
    "Horizon comparisons use **common forecast origins**. Compare normalized squared-error "
    "skill across horizons; lower raw RMSE on a shorter return horizon is not evidence of "
    "a better model. The primary horizon remains 30 calendar days. **Previous** adapts the "
    "original RF/Holt-Winters/AR methodology to each target; it is not the exact saved five-day model."
))


In [ ]:
display(Markdown(f"### Latest saved forecasts · cached origin {cached_origin:%d %B %Y}"))
forecast_table = latest_forecasts.copy()
forecast_table["target"] = forecast_table.experiment.map(HORIZONS)
forecast_table["origin"] = forecast_table.as_of_date.dt.strftime("%Y-%m-%d")
forecast_table["approximate_end"] = forecast_table.target_date_estimate.dt.strftime("%Y-%m-%d")
display(forecast_table[["target", "origin", "approximate_end", "predicted_return",
                         "cv_empirical_lower_80", "cv_empirical_upper_80", "close", "implied_close"]]
        .style.format({"predicted_return": "{:+.2%}", "cv_empirical_lower_80": "{:+.2%}",
                       "cv_empirical_upper_80": "{:+.2%}", "close": "{:.2f}", "implied_close": "{:.2f}"})
        .hide(axis="index"))
display(Markdown(
    "Projected endpoints are **weekday estimates** and do not project exchange holidays "
    "or future missing sessions. The displayed band uses the selected validation model's "
    "historical absolute-error quantile; it does not guarantee future 80% coverage."
))


In [ ]:
def render_dashboard(experiment=PRIMARY, save_path=None):
    if experiment not in HORIZONS:
        raise ValueError("Choose a saved MonthFu experiment.")
    rows = holdouts[experiment]
    latest = latest_forecasts.loc[latest_forecasts.experiment.eq(experiment)].iloc[0]
    summary = summaries[experiment]
    metrics = {row["model"]: row for row in summary["holdout_metrics"]}
    selected, previous = metrics["selected"], metrics["previous_transferred_ensemble"]
    background, ink, muted, coffee, teal = "#f6f2e9", "#292d2b", "#7d8988", "#ad6a37", "#417d79"
    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                         "axes.spines.top": False, "axes.spines.right": False,
                         "axes.spines.left": False, "axes.spines.bottom": False,
                         "axes.labelcolor": ink, "text.color": ink,
                         "xtick.color": muted, "ytick.color": muted}):
        fig = plt.figure(figsize=(15, 10.8), facecolor=background)
        grid = fig.add_gridspec(3, 2, height_ratios=[0.72, 1.6, 1.15],
                               hspace=0.56, wspace=0.30)
        card = fig.add_subplot(grid[0, 0])
        card.set_facecolor(background)
        card.axis("off")
        card.text(0, 0.92, "LATEST SAVED RETURN", color=muted, size=10, weight="bold")
        card.text(0, 0.43, f"{latest.predicted_return:+.2%}", color=coffee, size=35, weight="bold")
        card.text(0, 0.08,
                  f"Origin {latest.as_of_date:%d %b %Y} · approximate endpoint {latest.target_date_estimate:%d %b %Y}\n"
                  f"Empirical error band {latest.cv_empirical_lower_80:+.1%} to {latest.cv_empirical_upper_80:+.1%}",
                  color=muted, size=10, linespacing=1.5)

        score = fig.add_subplot(grid[0, 1])
        score.set_facecolor(background)
        score.barh(["Selected", "Adapted previous", "Zero return"],
                   [100*metrics[name]["rmse"] for name in
                    ["selected", "previous_transferred_ensemble", "zero"]],
                   color=[coffee, teal, "#bdc6c2"], height=0.55)
        score.invert_yaxis()
        score.set_title("Holdout return RMSE", loc="left", size=12, weight="bold")
        score.set_xlabel("Percentage points · lower is better", size=9)
        score.grid(axis="x", alpha=0.12)
        score.set_axisbelow(True)

        timeline = fig.add_subplot(grid[1, :])
        timeline.set_facecolor(background)
        timeline.plot(rows.Date, 100*rows.target_return, color="#a9b7b3", lw=1,
                      alpha=0.8, label="Observed monthly return")
        timeline.fill_between(rows.Date, 100*rows.empirical_lower_80, 100*rows.empirical_upper_80,
                              color=coffee, alpha=0.09, linewidth=0)
        timeline.plot(rows.Date, 100*rows.selected, color=coffee, lw=1.5,
                      label="Saved walk-forward forecast")
        timeline.axhline(0, color=muted, lw=0.8, alpha=0.6)
        timeline.set_ylabel("Return (%)")
        timeline.set_title("Quarterly walk-forward evaluation · recipe frozen before holdout",
                           loc="left", size=12, weight="bold", pad=12)
        timeline.grid(axis="y", alpha=0.12)
        timeline.legend(frameon=False, ncol=2, loc="upper left", fontsize=9)

        horizons = fig.add_subplot(grid[2, 0])
        horizons.set_facecolor(background)
        ordered = comparison.set_index("experiment").loc[list(HORIZONS)]
        x = np.arange(len(ordered))
        horizons.bar(x-0.18, 100*ordered.cv_skill_vs_zero, 0.36, color=teal, label="Selection CV")
        horizons.bar(x+0.18, 100*ordered.holdout_skill_vs_zero, 0.36, color=coffee, label="Common-date holdout")
        horizons.set_xticks(x, ["30 calendar\nPRIMARY", "21 sessions", "28 calendar", "21 calendar", "30 sessions"], fontsize=9)
        horizons.axhline(0, color=muted, lw=0.8)
        horizons.set_ylabel("Squared-error skill vs zero (%)")
        horizons.set_title("Horizon sensitivity", loc="left", size=12, weight="bold", pad=12)
        horizons.legend(frameon=False, ncol=2, fontsize=8, loc="upper left")
        horizons.grid(axis="y", alpha=0.12)
        horizons.set_axisbelow(True)

        annual = fig.add_subplot(grid[2, 1])
        annual.set_facecolor(background)
        years = yearly[experiment]
        for model, color, label in [("selected", coffee, "Selected"),
                                    ("previous_transferred_ensemble", teal, "Adapted previous")]:
            part = years.loc[years.model.eq(model)].sort_values("year")
            annual.plot(part.year, 100*part.rmse, marker="o", markersize=5,
                        color=color, lw=1.6, label=label)
        annual.set_xticks(sorted(years.year.unique()))
        annual.set_ylabel("RMSE (percentage points)")
        annual.set_title("Stability by forecast-origin year", loc="left", size=12, weight="bold", pad=12)
        annual.legend(frameon=False, ncol=2, fontsize=9)
        annual.grid(axis="y", alpha=0.12)

        fig.suptitle("ARABICA  /  MONTHFU", x=0.055, y=0.98, ha="left",
                     size=23, weight="bold", color=ink)
        fig.text(0.055, 0.938, HORIZONS[experiment] + " · cached research forecast",
                 size=12, color=muted)
        gain = 1-selected["rmse"]/previous["rmse"]
        fig.text(0.055, 0.024,
                 f"RMSE improvement vs adapted previous: {gain:+.1%}  ·  Direction: {selected['direction_accuracy']:.1%}\n"
                 "Historical holdout reused · overlapping targets · imperfect source vintages · no execution-cost claim",
                 size=9, color=muted, linespacing=1.5)
        fig.subplots_adjust(left=0.075, right=0.965, top=0.88, bottom=0.12)
        if save_path is not None:
            path = Path(save_path).resolve()
            if not path.is_relative_to(MONTHFU):
                raise ValueError("Save dashboard artifacts inside MonthFu.")
            path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(path, dpi=160, facecolor=background)
    return fig

DASHBOARD_PATH = ARTIFACTS / "monthfu_dashboard.png"
fig = render_dashboard(VIEW_EXPERIMENT, DASHBOARD_PATH)
display(fig)
plt.close(fig)


In [ ]:
display(Markdown("### Annual results for the selected view"))
annual_table = yearly[VIEW_EXPERIMENT]
annual_table = annual_table.loc[annual_table.model.isin(["selected", "previous_transferred_ensemble", "zero"])]
display(annual_table[["year", "model", "rows", "rmse", "mae", "direction_accuracy", "r2_vs_zero"]]
        .style.format({"rows": "{:,.0f}", "rmse": "{:.2%}", "mae": "{:.2%}",
                       "direction_accuracy": "{:.2%}", "r2_vs_zero": "{:+.2%}"})
        .hide(axis="index"))
summary = summaries[VIEW_EXPERIMENT]
interval = summary["block_comparisons"]["previous_transferred_ensemble"]["rmse_difference_95pct_block_interval"]
display(Markdown(
    f"**Selected minus adapted-previous RMSE:** 60-session block-bootstrap interval "
    f"**[{interval[0]:+.2%}, {interval[1]:+.2%}]**. Negative favors the selected model. "
    "Small average improvements can remain uncertain or unstable across years. "
    "Each experiment also saves 30/90-session block sensitivity and non-overlapping-origin diagnostics."
))


## Inspect saved forecasts and historical replay

Select a horizon and a saved evaluated date. **Replay** shows the exact stored out-of-sample score, the subsequently observed target return and its realized endpoint. **Latest cached forecast** shows the separately saved final-refit forecast.

The interactive controls are optional. Without widgets, edit `VIEW_EXPERIMENT` and `REPLAY_DATE` above, then rerun the view cells. Every historical view reads `holdout_predictions.csv`; no final fitted model is loaded.


In [ ]:
REPLAY_SOURCE = "saved_holdout_predictions.csv"

def replay_saved(experiment, origin):
    rows = holdouts[experiment]
    origin = pd.Timestamp(origin).normalize()
    selected = rows.loc[rows.Date.eq(origin)]
    if len(selected) != 1:
        raise ValueError("Choose exactly one saved evaluated holdout date.")
    return selected[["Date", "Close", "target_end_date", "selected", "target_return",
                     "empirical_lower_80", "empirical_upper_80"]].copy()

def display_saved_view(experiment=VIEW_EXPERIMENT, origin=None):
    if origin is None:
        selected = latest_forecasts.loc[latest_forecasts.experiment.eq(experiment)]
        display(Markdown(f"**Latest cached forecast · {HORIZONS[experiment]}**"))
        display(selected[["as_of_date", "target_date_estimate", "predicted_return",
                          "cv_empirical_lower_80", "cv_empirical_upper_80"]]
                .style.format({"predicted_return": "{:+.2%}", "cv_empirical_lower_80": "{:+.2%}",
                               "cv_empirical_upper_80": "{:+.2%}"}).hide(axis="index"))
    else:
        replay = replay_saved(experiment, origin)
        display(Markdown(f"**Saved walk-forward replay · {HORIZONS[experiment]}**"))
        display(replay.style.format({"Close": "{:.2f}", "selected": "{:+.2%}",
                                     "target_return": "{:+.2%}", "empirical_lower_80": "{:+.2%}",
                                     "empirical_upper_80": "{:+.2%}"}).hide(axis="index"))
        display(Markdown("Historical prediction copied directly from saved evaluated scores; final refitted weights are not used."))

display_saved_view(VIEW_EXPERIMENT, REPLAY_DATE)

try:
    import ipywidgets as widgets
    horizon_control = widgets.Dropdown(options=[(label, key) for key, label in HORIZONS.items()],
                                       value=VIEW_EXPERIMENT, description="Horizon:", layout={"width": "440px"})
    mode_control = widgets.Dropdown(options=["Latest cached forecast", "Saved walk-forward replay"],
                                    description="View:", layout={"width": "440px"})
    date_control = widgets.Dropdown(options=holdouts[VIEW_EXPERIMENT].Date.dt.strftime("%Y-%m-%d").tolist(),
                                    description="Origin:", layout={"width": "440px"})
    date_control.value = date_control.options[-1]
    date_control.disabled = True
    view_output = widgets.Output()

    def update_controls(change=None):
        if change and change.get("owner") is horizon_control:
            options = holdouts[horizon_control.value].Date.dt.strftime("%Y-%m-%d").tolist()
            date_control.options = options
            date_control.value = options[-1]
        date_control.disabled = mode_control.value != "Saved walk-forward replay"
        with view_output:
            view_output.clear_output(wait=True)
            origin = None if date_control.disabled else date_control.value
            display_saved_view(horizon_control.value, origin)

    horizon_control.observe(update_controls, names="value")
    mode_control.observe(update_controls, names="value")
    date_control.observe(update_controls, names="value")
    display(widgets.VBox([horizon_control, mode_control, date_control, view_output]))
    update_controls()
except ImportError:
    print("ipywidgets is optional. The saved view above works without it.")


## Reproduce and update

From the **repository root**:

```sh
# Fit all five fixed horizon experiments, then generate comparison/forecast tables.
.venv/bin/python MonthFu/scripts/run_experiments.py --jobs 2

# Train only the primary target.
.venv/bin/python MonthFu/scripts/train.py --horizon 30 --horizon-unit calendar

# Rebuild the latest cached-origin forecast with saved fitted weights.
.venv/bin/python MonthFu/scripts/predict.py

# Replay one actual evaluated score; this does not load final refitted weights.
.venv/bin/python MonthFu/scripts/predict.py --replay-date 2024-06-03

# Verify every saved metric table without starting a Jupyter kernel.
.venv/bin/python MonthFu/scripts/check_notebook.py --artifacts-only

# Execute this notebook and render/check the dashboard in a permitted kernel environment.
.venv/bin/python MonthFu/scripts/check_notebook.py
```

The acquisition scripts remain in their original locations. Refresh source caches before retraining when a newer market origin is required. The forecast script itself does not fetch market data.

Read [feature timing and engineering](docs/FEATURES.md), [COT alignment](docs/COT_ALIGNMENT.md), [research review](docs/RESEARCH_REVIEW.md) and the generated [results](RESULTS.md). Per-horizon folders preserve the frozen selection, CV forecasts, quarterly refit audits, release audits, fitted bundle, evaluation scores and report.


In [ ]:
# Verify every replay is exactly the saved score, with no historical inference.
replay_count = 0
for experiment, rows in holdouts.items():
    for position in [0, len(rows)//2, len(rows)-1]:
        source = rows.iloc[position]
        replay = replay_saved(experiment, source.Date).iloc[0]
        for column in ["selected", "target_return", "empirical_lower_80", "empirical_upper_80"]:
            assert float(replay[column]) == float(source[column])
        assert replay.target_end_date == source.target_end_date
        replay_count += 1
assert REPLAY_SOURCE == "saved_holdout_predictions.csv"
assert DASHBOARD_PATH.exists()
NOTEBOOK_CHECKS = {
    "horizons": len(holdouts),
    "cached_origin": str(cached_origin.date()),
    "saved_replay_verified": True,
    "replay_examples_verified": replay_count,
    "dashboard": str(DASHBOARD_PATH.relative_to(ROOT)),
}
print("MONTHFU_NOTEBOOK_CHECKS=" + json.dumps(NOTEBOOK_CHECKS, sort_keys=True))
